# AI-Voxel Dataset Factory v4 — ML / Inverse-Design Ready

Threadripper 3970X + Dual RTX 3090 adaptive scheduling, disk-first resume, continuous latent-controlled Voxel generation, STL/DLP/descriptor export, and permanent ML dataset registry.

**핵심:** Descriptor-LHS에 선정되지 않은 pool 구조도 `04_ml_dataset/all_structures.csv`에 영구 보존되어 Model 1 학습에 사용됩니다. `seed`는 provenance로만 저장되고 ML/inverse-design control에서는 제외합니다.


In [ ]:
# ============================================================
# CELL 1 — ALL SETTINGS + PERSISTENT RUN CONTEXT
# 30,000-candidate / 96-final-sample configuration
# ============================================================
# [중요]
# - 새 연구 run 시작: START_NEW_RESULT_RUN = True 후 CELL 1 실행
#   → BASE_DIR 아래 Result_YYYYMMDD_HHMMSS 폴더를 자동 생성합니다.
# - Kernel restart / 중단 후 재개: CELL 1을 다시 실행하지 말고 중단된 stage 셀부터 실행합니다.
# - 부득이하게 CELL 1부터 재개해야 하면 START_NEW_RESULT_RUN = False로 바꾸면
#   .ai_voxel_active_run.json이 가리키는 기존 Result 폴더를 다시 사용합니다.
from pathlib import Path
from datetime import datetime
import json, os, hashlib

# =====================================================================
# A. IMPORT WORKSPACE / EXPORT RESULT ROOT
# =====================================================================
BASE_DIR = Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ACTIVE_POINTER_FILENAME = ".ai_voxel_active_run.json"
RESULT_PREFIX = "Result"
START_NEW_RESULT_RUN = True

if START_NEW_RESULT_RUN:
    RUN_NAME = f"{RESULT_PREFIX}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    RUN_DIR = BASE_DIR / RUN_NAME
else:
    _ptr = BASE_DIR / ACTIVE_POINTER_FILENAME
    if not _ptr.exists():
        raise FileNotFoundError(f"기존 run pointer가 없습니다: {_ptr}. 새 run은 START_NEW_RESULT_RUN=True로 실행하세요.")
    _active = json.loads(_ptr.read_text(encoding="utf-8"))
    RUN_DIR = Path(_active["run_dir"])
    RUN_NAME = RUN_DIR.name

# ---------- durable/resume ----------
RESUME_SKIP_COMPLETED = True          # STL/descriptor 등 완료된 item은 재실행 시 자동 skip
SAVE_PICKLE_CHECKPOINTS = True        # DataFrame checkpoint를 CSV + pickle로 함께 저장
FORCE_RERUN_STAGES = []               # 예: ["04_pool_generation", "06_pool_descriptor"]
LARGE_POOL_LOG_FLUSH_EVERY_N = 25    # 30k pool에서 aggregate CSV를 매 item마다 다시 쓰지 않고 25개마다 flush
# 30,000행 전체 XLSX는 매우 느리고 파일도 커지므로 full data는 CSV/Parquet가 source-of-truth입니다.
EXPORT_FULL_CANDIDATE_XLSX = False
CANDIDATE_XLSX_PREVIEW_ROWS = 5_000

# =====================================================================
# B. ML / INVERSE-DESIGN COMPATIBILITY
# =====================================================================
GENERATOR_VERSION = "v4_latent_controlled"
LATENT_DIM = 16
# generate | import | inverse_design | active_sampling
CANDIDATE_SOURCE = "generate"
CANDIDATE_IMPORT_FILE = ""
INVERSE_DESIGN_CANDIDATE_FILE = ""
ACTIVE_SAMPLING_CANDIDATE_FILE = ""
SELECT_ALL_IMPORTED_CANDIDATES = True
INCLUDE_LEGACY_SEED_MODES_IN_NEW_SAMPLING = False

# =====================================================================
# C. SAMPLING — 변수 의미를 명확히 재정의
# =====================================================================
RANDOM_SEED = 42

# 전체 후보 Voxel 구조 개수입니다.
# USE_GENERATOR_PARAMETER_LHS=True이면 "랜덤 30,000개"가 아니라
# 생성인자 공간을 LHS로 넓게 채운 30,000개 후보를 뜻합니다.
N_CANDIDATE_STRUCTURES = 30_000
N_RANDOM_STRUCTURES = N_CANDIDATE_STRUCTURES   # 기존 runtime 호환용 alias

# 30,000개 후보의 구조인자 공간에서 최종 제작/정밀평가 대상으로 선정할 개수입니다.
N_LHS_SELECTED_STRUCTURES = 96
N_FINAL_SAMPLES = N_LHS_SELECTED_STRUCTURES    # 기존 runtime 호환용 alias

# True: target_vf, thickness, anisotropy, sigma, latent z 등 "생성인자 공간" 자체를
# Latin Hypercube Sampling으로 균등하게 분산시켜 30,000개 candidate parameter를 만듭니다.
USE_GENERATOR_PARAMETER_LHS = True

# True: descriptor를 추출한 전체 pool을 PCA 구조인자 공간으로 축약한 뒤,
# 그 공간에서 LHS target에 가장 가까운 서로 다른 실제 구조 96개를 선정합니다.
USE_DESCRIPTOR_LHS_SELECTION = True

# Descriptor-LHS에서 수천 개 구조인자를 몇 개의 PCA 축으로 축약할지 지정합니다.
# 10이면 구조인자 변화의 주요 방향을 10차원 공간으로 만들어 LHS selection을 수행합니다.
DESCRIPTOR_LHS_PCA_COMPONENTS = 10
DESCRIPTOR_LHS_PCA_SOLVER = "randomized"       # 30k 대규모 pool용 빠른 PCA

# 특정 descriptor가 30,000개 중 25%를 초과해 NaN이면 selection feature에서 제외합니다.
DESCRIPTOR_MAX_MISSING_FRACTION = 0.25

# 구조 간 변화가 사실상 없는 descriptor(표준편차 <= 1e-12)를 selection에서 제외합니다.
DESCRIPTOR_MIN_STD = 1e-12

# =====================================================================
# D. GEOMETRY / RESOLUTION — 변수 의미
# =====================================================================
# Voxel 구조의 실제 cubic boundary 한 변 길이 [mm]. 30 mm cube를 생성합니다.
BOUNDARY_SIZE_MM = 30.0

# 최종 선정 96개 구조를 재생성할 목표 voxel pitch [mm].
# 30/0.03 = 축당 1000 voxel → 최종 1000^3 grid입니다.
VOXEL_SIZE_MM = 0.03

# True: 30,000개 전체 후보는 더 가벼운 screening voxel size로 먼저 생성/분석하고,
# 최종 선정 구조만 0.03 mm로 재생성합니다.
USE_SCREENING_RESOLUTION = True

# 후보 pool용 voxel pitch [mm]. 30/0.10 = 축당 300 voxel → 300^3 grid입니다.
SCREENING_VOXEL_SIZE_MM = 0.10

# True: LHS로 선정된 96개를 screening STL 복사로 끝내지 않고
# 동일 생성인자를 사용해 VOXEL_SIZE_MM=0.03 mm에서 다시 생성합니다.
REGENERATE_SELECTED_AT_FINAL_RESOLUTION = True

# marching cubes의 grid sampling step입니다. 값이 클수록 STL 변환이 빠르고 mesh가 거칠어집니다.
# screening=2: 30k pool의 STL 생성속도/파일용량 절감.
MARCHING_CUBES_STEP_SCREENING = 2
# final=1: 선정 96개는 모든 voxel layer를 사용해 최대한 정밀한 STL 생성.
MARCHING_CUBES_STEP_FINAL = 1

# 1000^3 같은 고메모리 final grid 생성을 허용할지 결정합니다.
# False이면 runtime이 grid_n>=800을 안전상 차단합니다.
ALLOW_HIGH_MEMORY_FINAL_GRID = True

# max_thickness_mm를 넘는 과도하게 두꺼운 영역을 morphology 기반으로 제한할지 여부입니다.
ENABLE_MAX_THICKNESS_TRIM = True
# 이 trim은 추가 distance/morphology 메모리를 많이 사용하므로 grid_n<=520에서만 수행합니다.
# 따라서 300^3 screening에는 적용될 수 있지만 1000^3 final에서는 메모리 안전을 위해 skip됩니다.
MAX_THICKNESS_TRIM_GRID_LIMIT = 520

# =====================================================================
# E. CPU/GPU — Threadripper 3970X + dual RTX 3090 adaptive/shared workstation
# =====================================================================
CPU_PHYSICAL_CORES = 32
CPU_LOGICAL_THREADS = os.cpu_count() or 64
ADAPTIVE_CPU_SCHEDULING = True
CPU_RESERVE_PHYSICAL_CORES = 4
CPU_PROBE_INTERVAL_SEC = 0.25
DESCRIPTOR_CPU_WORKERS = 28
DESCRIPTOR_CPU_WORKERS_MAX = 28
DESCRIPTOR_CPU_WORKERS_MIN = 16
DESCRIPTOR_INNER_BLAS_THREADS = 1
DLP_PARALLEL_WORKERS = 24
DLP_PARALLEL_BACKEND = "loky"

ENABLE_NVIDIA_GPU = True
ADAPTIVE_GPU_SCHEDULING = True
GPU_PREFERRED_NAME = "RTX 3090"
GPU_REQUIRED_NAME_SUBSTRING = "RTX 3090"
GPU_CANDIDATE_IDS = [0, 1]
GPU_PREFERRED_ORDER = [1, 0]
GPU0_SHARED_WORKLOAD_PENALTY = 0.08
GPU_MIN_FREE_VRAM_GB = 8.0
GPU_HARD_MIN_FREE_VRAM_GB = 4.0
GPU_BUSY_UTILIZATION_PCT = 72.0
GPU_BUSY_MEMORY_USED_FRACTION = 0.72
GPU_ALLOW_LEAST_BUSY_FALLBACK = True
GPU_SCORE_FREE_WEIGHT = 0.70
GPU_SCORE_UTIL_WEIGHT = 0.30
GPU_MAX_GRID_N = 720
GENERATION_PARALLEL_SCREENING_WORKERS = 2
GENERATION_PARALLEL_FINAL_WORKERS = 1
GENERATION_CPU_FALLBACK_WORKERS = max(2, min(8, CPU_PHYSICAL_CORES // 4))
GPU_STATUS_CACHE_SEC = 1.0
DESCRIPTOR_ENABLE_PARALLEL = True
DESCRIPTOR_ENABLE_GPU = True

# =====================================================================
# F. VOXEL DESIGN RANGES
# =====================================================================
VOXEL_MODES = ["latent_periodic_isotropic", "latent_periodic_orthotropic", "latent_stochastic"]
GEN_PARAM_RANGES = {
    "target_vf": [0.12, 0.68], "min_thickness_mm": [0.09, 0.60],
    "min_hole_size_mm": [0.09, 0.75], "max_thickness_mm": [0.60, 3.00],
    "closing_radius_mm": [0.00, 0.12], "opening_radius_mm": [0.00, 0.12],
    "connectivity_bridge_radius_mm": [0.06, 0.18], "contact_surface_depth_mm": [0.03, 0.15],
    "num_fourier_terms": [4, 28], "fourier_k_max": [3, 8],
    "anisotropy_z": [0.65, 1.80], "sigma_mm": [0.12, 1.20],
}
ENFORCE_CONTACT_FACE_SYMMETRY=True
STRICT_GLOBAL_SYMMETRY=True
STRICT_SYMMETRY_METHOD="score_threshold"
FORCE_CONNECTED_LATTICE_VOXEL=True
CONNECTIVITY_REPAIR_MODE="bridge"
CONNECTIVITY_MIN_COMPONENT_VOXELS=1
CONNECTIVITY_MAX_BRIDGES=200
CONNECTIVITY_RETRY_AFTER_CONTACT_SYMMETRY=True
VF_TOLERANCE=0.05

# =====================================================================
# G. DLP
# =====================================================================
GENERATE_DLP_FOR_POOL=False
GENERATE_DLP_FOR_SELECTED=True
DLP_PIXEL_SIZE_MM=0.065
DLP_LAYER_HEIGHT_MM=0.10
DLP_CANVAS_W=1920; DLP_CANVAS_H=1080; DLP_PART_PX=461; DLP_FIXED_LAYER_COUNT=300
DLP_TEMPLATE_SLICE_DIR = BASE_DIR / "Slice Template" / "Compression.slice"
COPY_DLP_TEMPLATE_ASSETS_IF_AVAILABLE=True
DLP_OVERWRITE=True
DLP_VERBOSE=False

# =====================================================================
# H. DESCRIPTOR v6
# =====================================================================
RUN_DESCRIPTOR_FOR_POOL=True
RUN_DESCRIPTOR_FOR_SELECTED=True
DESCRIPTOR_CROP_CUBE_SIZE_MM=BOUNDARY_SIZE_MM
DESCRIPTOR_IMAGE_PIXELS_PER_SIDE=1000
DESCRIPTOR_LAYER_SLICE_PERCENT=5
DESCRIPTOR_REFERENCE_VOXEL_SIZE_MM=0.20
POOL_DESCRIPTOR_PROFILE="screening"
FINAL_DESCRIPTOR_PROFILE="full"
SAVE_DESCRIPTOR_RAW_SLICE_FILES_POOL=True
SAVE_DESCRIPTOR_RAW_SLICE_FILES_FINAL=True
SAVE_DESCRIPTOR_DEBUG_IMAGES_POOL=False
SAVE_DESCRIPTOR_DEBUG_IMAGES_FINAL=False

# =====================================================================
# I. EXPORT FOLDER STRUCTURE — 모든 workflow 결과는 Result_timestamp 안에 저장
# =====================================================================
STATE_DIR=RUN_DIR/"00_state"
STAGE_DIR=STATE_DIR/"stages"
CHECKPOINT_DIR=STATE_DIR/"checkpoints"
RUNTIME_DIR=RUN_DIR/"_runtime"
ENGINE_DIR=RUN_DIR/"_descriptor_engine"

CANDIDATE_ROOT=RUN_DIR/"01_Candidate_Parameters"
POOL_ROOT=RUN_DIR/"02_Voxel_Pool"
POOL_STL_DIR=POOL_ROOT/"STL"
POOL_DLP_RAW_DIR=POOL_ROOT/"DLP_raw"
POOL_DLP_DIR=POOL_ROOT/"DLP_slice"
POOL_DESCRIPTOR_DIR=POOL_ROOT/"Descriptor"

SELECTION_ROOT=RUN_DIR/"03_Descriptor_LHS_Selection"
SELECTED_ROOT=RUN_DIR/"04_Selected_Final"
SELECTED_STL_DIR=SELECTED_ROOT/"STL"
SELECTED_DLP_RAW_DIR=SELECTED_ROOT/"DLP_raw"
SELECTED_DLP_DIR=SELECTED_ROOT/"DLP_slice"
SELECTED_DESCRIPTOR_DIR=SELECTED_ROOT/"Descriptor"

TABLE_DIR=RUN_DIR/"05_Tables"
ML_ROOT=RUN_DIR/"06_ML_Dataset"
ML_DATA_ROOT=ML_ROOT
ML_PROJECT_ROOT=ML_ROOT/"Project_State"
MODEL_ROOT=RUN_DIR/"07_Models"
INVERSE_ROOT=RUN_DIR/"08_Inverse_Optimization"
ACTIVE_ROOT=RUN_DIR/"09_Active_Sampling"
VALIDATION_ROOT=RUN_DIR/"10_Validation"

# BASE_DIR의 pointer 파일은 최신 Result 폴더를 찾기 위한 작은 control file일 뿐이며,
# 실제 연구 결과/모델/데이터는 모두 RUN_DIR(Result_timestamp) 안에 저장됩니다.
ML_PROJECT_POINTER=BASE_DIR/".ai_voxel_ml_project.json"

POOL_MASTER_XLSX=TABLE_DIR/"Pool_Structural_Descriptors.xlsx"
SELECTED_MASTER_XLSX=TABLE_DIR/"Selected_Final_Structural_Descriptors.xlsx"
SELECTION_XLSX=SELECTION_ROOT/"Descriptor_LHS_Selected_Samples.xlsx"

for p in [RUN_DIR,STATE_DIR,STAGE_DIR,CHECKPOINT_DIR,RUNTIME_DIR,ENGINE_DIR,CANDIDATE_ROOT,
          POOL_STL_DIR,POOL_DLP_RAW_DIR,POOL_DLP_DIR,POOL_DESCRIPTOR_DIR,SELECTION_ROOT,
          SELECTED_STL_DIR,SELECTED_DLP_RAW_DIR,SELECTED_DLP_DIR,SELECTED_DESCRIPTOR_DIR,
          TABLE_DIR,ML_ROOT,ML_PROJECT_ROOT,MODEL_ROOT,INVERSE_ROOT,ACTIVE_ROOT,VALIDATION_ROOT]:
    p.mkdir(parents=True,exist_ok=True)

# only JSON-serializable settings are persisted; every later cell reloads this file.
settings={k:v for k,v in globals().copy().items() if k.isupper() and not k.startswith('_')}
def enc(v):
    if isinstance(v,Path): return str(v)
    if isinstance(v,tuple): return [enc(x) for x in v]
    if isinstance(v,list): return [enc(x) for x in v]
    if isinstance(v,dict): return {str(k):enc(x) for k,x in v.items()}
    if isinstance(v,(str,int,float,bool)) or v is None: return v
    return str(v)
settings={k:enc(v) for k,v in settings.items()}
config_path=STATE_DIR/"config.json"
config_path.write_text(json.dumps(settings,indent=2,ensure_ascii=False),encoding='utf-8')
hash_exclude={'FORCE_RERUN_STAGES','RESUME_SKIP_COMPLETED','SAVE_PICKLE_CHECKPOINTS','START_NEW_RESULT_RUN'}
hash_payload={k:v for k,v in settings.items() if k not in hash_exclude}
config_hash=hashlib.sha256(json.dumps(hash_payload,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
active={"run_name":RUN_NAME,"run_dir":str(RUN_DIR),"config_path":str(config_path),"config_sha256":config_hash}
(BASE_DIR/ACTIVE_POINTER_FILENAME).write_text(json.dumps(active,indent=2,ensure_ascii=False),encoding='utf-8')
print("Active RUN_DIR:",RUN_DIR)
print("Config SHA256:",config_hash[:16],"...")
print(f"Candidates: {N_CANDIDATE_STRUCTURES:,} | Descriptor-LHS final: {N_LHS_SELECTED_STRUCTURES:,}")
print("Final grid:",round(BOUNDARY_SIZE_MM/VOXEL_SIZE_MM),"^3 | Screening grid:",round(BOUNDARY_SIZE_MM/SCREENING_VOXEL_SIZE_MM),"^3")
print("\nWorkflow export folders:")
for p in [CANDIDATE_ROOT,POOL_ROOT,SELECTION_ROOT,SELECTED_ROOT,TABLE_DIR,ML_ROOT,MODEL_ROOT,INVERSE_ROOT,ACTIVE_ROOT,VALIDATION_ROOT]:
    print(" -",p)


In [ ]:
# ============================================================
# CELL 2 — ONE-TIME V4 RUNTIME/ENGINE DEPLOY + HARDWARE PROBE
# ============================================================
# Bundle 안의 최신 runtime/descriptor engine을 현재 Result 폴더에 복사합니다.
# 이후 Kernel이 재시작되어도 Result/_runtime 및 Result/_descriptor_engine 파일을 다시 읽을 수 있습니다.
import os,sys,json,importlib,shutil
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 first")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"; ENGINE_DIR=RUN_DIR/"_descriptor_engine"
RUNTIME_DIR.mkdir(parents=True,exist_ok=True); ENGINE_DIR.mkdir(parents=True,exist_ok=True)

# 현재 Notebook과 함께 압축해제된 source file 위치를 자동 탐색합니다.
_required=["ai_voxel_runtime_dual3090_v4.py","descriptor_library.py","stl_pipeline_driver.py"]
_search_roots=[Path.cwd(), Path.cwd()/"AI_Voxel_ML_InverseDesign_v4_30K96", BASE_DIR]
source_dir=None
for root in _search_roots:
    if root.exists() and all((root/f).is_file() for f in _required):
        source_dir=root; break
if source_dir is None:
    # 마지막 fallback: 현재 cwd 아래 몇 단계만 검색
    for cand in Path.cwd().glob("**/ai_voxel_runtime_dual3090_v4.py"):
        root=cand.parent
        if all((root/f).is_file() for f in _required): source_dir=root; break
if source_dir is None:
    raise FileNotFoundError("Bundle source files를 찾지 못했습니다. Notebook과 *.py 파일들을 같은 폴더에 두세요.")

shutil.copy2(source_dir/"ai_voxel_runtime_dual3090_v4.py",RUNTIME_DIR/"ai_voxel_runtime_dual3090_v4.py")
shutil.copy2(source_dir/"descriptor_library.py",ENGINE_DIR/"descriptor_library.py")
shutil.copy2(source_dir/"stl_pipeline_driver.py",ENGINE_DIR/"stl_pipeline_driver.py")

for pp in [str(RUNTIME_DIR),str(ENGINE_DIR)]:
    if pp not in sys.path: sys.path.insert(0,pp)
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)
hw=rt.hardware_report()
rt.atomic_json(rt.STATE_DIR/"hardware.json",hw)
rt.save_stage_manifest("02_runtime_hardware","completed",[
    RUNTIME_DIR/"ai_voxel_runtime_dual3090_v4.py",
    ENGINE_DIR/"descriptor_library.py",
    ENGINE_DIR/"stl_pipeline_driver.py",
    rt.STATE_DIR/"hardware.json"],hw)
print("Bundle source:",source_dir)
print(json.dumps(hw,indent=2,ensure_ascii=False))
print("Adaptive GPU snapshot:",json.dumps(rt.adaptive_gpu_snapshot(),indent=2,ensure_ascii=False))


## Durable stages
모든 stage는 disk checkpoint를 사용합니다. `CANDIDATE_SOURCE`를 `inverse_design` 또는 `active_sampling`으로 바꾸면 후속 Notebook의 CSV를 그대로 입력받아 최종 Voxel/STL/DLP/descriptor를 생성합니다.


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="03_candidate_parameters"; out_csv=rt.CANDIDATE_ROOT/"Candidate_Generation_Parameters.csv"
if rt.stage_can_resume(stage,[out_csv]):
    candidate_df=rt.load_df_checkpoint("candidate_parameters") if (rt.CHECKPOINT_DIR/"candidate_parameters.csv").exists() else rt.pd.read_csv(out_csv)
    print("RESUME: loaded",len(candidate_df),"candidates from disk")
else:
    candidate_df=rt.build_voxel_candidate_table(n_total=rt.N_RANDOM_STRUCTURES,seed=rt.RANDOM_SEED,use_lhs=rt.USE_GENERATOR_PARAMETER_LHS)
    rt.save_df_checkpoint(candidate_df,"candidate_parameters")
    rt.save_stage_manifest(stage,"completed",[out_csv,rt.TABLE_DIR/"Candidate_Generation_Parameters.xlsx",rt.CHECKPOINT_DIR/"candidate_parameters.csv"],{"rows":len(candidate_df)})
display(candidate_df.head())


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="04_pool_generation"; candidate_df=rt.load_df_checkpoint("candidate_parameters")
pool_vs=rt.SCREENING_VOXEL_SIZE_MM if rt.USE_SCREENING_RESOLUTION else rt.VOXEL_SIZE_MM
force=not rt.stage_can_resume(stage,[rt.TABLE_DIR/f"generation_log_{'screening' if rt.USE_SCREENING_RESOLUTION else 'final_resolution_pool'}.csv"])
log=rt.run_voxel_generation(candidate_df,pool_vs,rt.POOL_STL_DIR,"screening" if rt.USE_SCREENING_RESOLUTION else "final_resolution_pool",force=force)
if not (log["status"].astype(str).str.upper()=="FAILED").any(): rt.save_stage_manifest(stage,"completed",[rt.TABLE_DIR/f"generation_log_{'screening' if rt.USE_SCREENING_RESOLUTION else 'final_resolution_pool'}.csv",rt.POOL_STL_DIR],{"rows":len(log),"voxel_size_mm":pool_vs})
else: rt.save_stage_manifest(stage,"partial",metadata={"failed":int((log['status'].astype(str).str.upper()=='FAILED').sum())})
display(log.head())


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="05_pool_dlp"; gen=rt.load_df_checkpoint("generation_log_screening" if rt.USE_SCREENING_RESOLUTION else "generation_log_final_resolution_pool")
force=not rt.stage_can_resume(stage,[rt.TABLE_DIR/"DLP_log_pool.csv"])
log=rt.run_dlp_batch(rt.POOL_STL_DIR,rt.POOL_DLP_RAW_DIR,rt.POOL_DLP_DIR,enabled=rt.GENERATE_DLP_FOR_POOL,force=force,log_tag="pool")
if not rt.GENERATE_DLP_FOR_POOL:
    rt.save_stage_manifest(stage,"completed",metadata={"skipped_by_config":True})
elif len(log) and not (log['status'].astype(str).str.upper()=='FAILED').any(): rt.save_stage_manifest(stage,"completed",[rt.TABLE_DIR/"DLP_log_pool.csv",rt.POOL_DLP_DIR],{"rows":len(log)})
else: rt.save_stage_manifest(stage,"partial")
display(log.head() if len(log) else log)


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="06_pool_descriptor"; _=rt.load_df_checkpoint("generation_log_screening" if rt.USE_SCREENING_RESOLUTION else "generation_log_final_resolution_pool")
log_csv=rt.TABLE_DIR/"descriptor_run_log_pool.csv"; force=not rt.stage_can_resume(stage,[log_csv])
log=rt.run_descriptor_batch(rt.POOL_STL_DIR,rt.POOL_DESCRIPTOR_DIR,enabled=rt.RUN_DESCRIPTOR_FOR_POOL,profile=rt.POOL_DESCRIPTOR_PROFILE,save_raw_slice_files=rt.SAVE_DESCRIPTOR_RAW_SLICE_FILES_POOL,save_debug_images=rt.SAVE_DESCRIPTOR_DEBUG_IMAGES_POOL,force=force,log_tag="pool")
if not rt.RUN_DESCRIPTOR_FOR_POOL: rt.save_stage_manifest(stage,"completed",metadata={"skipped_by_config":True})
elif len(log) and not (log['status'].astype(str).str.upper()=='FAILED').any(): rt.save_stage_manifest(stage,"completed",[log_csv,rt.POOL_DESCRIPTOR_DIR],{"rows":len(log),"adaptive_gpu":True,"gpu_policy":rt.adaptive_gpu_snapshot().get("policy",{}),"cpu_worker_policy":[rt.DESCRIPTOR_CPU_WORKERS_MIN,rt.DESCRIPTOR_CPU_WORKERS_MAX]})
else: rt.save_stage_manifest(stage,"partial")
display(log.head() if len(log) else log)


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="07_pool_master"; candidate_df=rt.load_df_checkpoint("candidate_parameters"); desc_log=rt.load_df_checkpoint("descriptor_log_pool")
pool_vs=rt.SCREENING_VOXEL_SIZE_MM if rt.USE_SCREENING_RESOLUTION else rt.VOXEL_SIZE_MM
master,desc_cols,catalog=rt.collect_descriptor_master(candidate_df,rt.POOL_DESCRIPTOR_DIR,rt.POOL_MASTER_XLSX,voxel_size_mm=pool_vs,stage="screening" if rt.USE_SCREENING_RESOLUTION else "final_resolution_pool",run_log=desc_log)
rt.save_df_checkpoint(master,"pool_master"); rt.atomic_json(rt.CHECKPOINT_DIR/"pool_descriptor_columns.json",desc_cols); rt.save_stage_manifest(stage,"completed",[rt.POOL_MASTER_XLSX,rt.POOL_MASTER_XLSX.with_suffix('.csv'),rt.CHECKPOINT_DIR/"pool_descriptor_columns.json"],{"rows":len(master),"descriptor_columns":len(desc_cols)})
display(master.head())


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation");ptr=BASE_DIR/".ai_voxel_active_run.json"
active=json.loads(ptr.read_text(encoding="utf-8"));RUN_DIR=Path(active["run_dir"]);os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime";sys.path.insert(0,str(RUNTIME_DIR)) if str(RUNTIME_DIR) not in sys.path else None
import ai_voxel_runtime_dual3090_v4 as rt;rt=importlib.reload(rt)
stage="08_descriptor_selection";master=rt.load_df_checkpoint("pool_master");desc_cols=json.loads((rt.CHECKPOINT_DIR/"pool_descriptor_columns.json").read_text(encoding="utf-8"))
if str(rt.CANDIDATE_SOURCE).lower()!='generate' and bool(rt.SETTINGS.get('SELECT_ALL_IMPORTED_CANDIDATES',True)):
    selected=master.copy();selected.insert(1,'selection_method',f'all_{rt.CANDIDATE_SOURCE}_candidates');selected.insert(2,'selection_rank',range(1,len(selected)+1))
    ids=selected['sample_id'].astype(str).tolist();state={'selected_ids':ids,'method':f'all_{rt.CANDIDATE_SOURCE}_candidates','n_selected':len(ids)}
    rt.atomic_csv(selected,rt.CHECKPOINT_DIR/'selected_samples.csv');rt.atomic_json(rt.CHECKPOINT_DIR/'selection_state.json',state)
else:
    selected,scores,used=rt.select_samples_from_descriptor_space(master,desc_cols,n_select=rt.N_FINAL_SAMPLES);state=rt.save_selection_model(master,desc_cols,selected)
rt.save_df_checkpoint(selected,"selected_pool_rows");rt.save_stage_manifest(stage,"completed",[rt.CHECKPOINT_DIR/"selection_state.json",rt.CHECKPOINT_DIR/"selected_samples.csv"],state)
print("Selected IDs:",state['selected_ids']);display(selected[["sample_id","selection_method","selection_rank"]].head(30))


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="09_final_generation"; candidate_df=rt.load_df_checkpoint("candidate_parameters"); state=json.loads((rt.CHECKPOINT_DIR/"selection_state.json").read_text(encoding="utf-8")); ids=list(map(str,state['selected_ids'])); sub=candidate_df[candidate_df['candidate_id'].astype(str).isin(set(ids))].copy()
if rt.REGENERATE_SELECTED_AT_FINAL_RESOLUTION:
    force=not rt.stage_can_resume(stage,[rt.TABLE_DIR/"generation_log_final.csv"]); log=rt.run_voxel_generation(sub,rt.VOXEL_SIZE_MM,rt.SELECTED_STL_DIR,"final",selected_ids=ids,force=force)
else:
    rows=[]
    for cid in ids:
        src=rt.POOL_STL_DIR/f"{cid}.stl"; dst=rt.SELECTED_STL_DIR/f"{cid}.stl"; rt.shutil.copy2(src,dst); rows.append({'candidate_id':cid,'status':'copied_from_pool','stl_path':str(dst)})
    log=rt.pd.DataFrame(rows); rt.save_df_checkpoint(log,'generation_log_final')
rt.save_stage_manifest(stage,"completed",[rt.TABLE_DIR/"generation_log_final.csv",rt.SELECTED_STL_DIR],{"rows":len(log)}); display(log.head())


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="10_final_dlp"; _=rt.load_df_checkpoint("generation_log_final"); log_csv=rt.TABLE_DIR/"DLP_log_final.csv"; force=not rt.stage_can_resume(stage,[log_csv]); log=rt.run_dlp_batch(rt.SELECTED_STL_DIR,rt.SELECTED_DLP_RAW_DIR,rt.SELECTED_DLP_DIR,enabled=rt.GENERATE_DLP_FOR_SELECTED,force=force,log_tag="final")
if not rt.GENERATE_DLP_FOR_SELECTED or (len(log) and not (log['status'].astype(str).str.upper()=='FAILED').any()): rt.save_stage_manifest(stage,"completed",[log_csv,rt.SELECTED_DLP_DIR] if rt.GENERATE_DLP_FOR_SELECTED else [],{"rows":len(log)})
else: rt.save_stage_manifest(stage,"partial")
display(log.head() if len(log) else log)


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="11_final_descriptor"; _=rt.load_df_checkpoint("generation_log_final"); log_csv=rt.TABLE_DIR/"descriptor_run_log_final.csv"; force=not rt.stage_can_resume(stage,[log_csv]); log=rt.run_descriptor_batch(rt.SELECTED_STL_DIR,rt.SELECTED_DESCRIPTOR_DIR,enabled=rt.RUN_DESCRIPTOR_FOR_SELECTED,profile=rt.FINAL_DESCRIPTOR_PROFILE,save_raw_slice_files=rt.SAVE_DESCRIPTOR_RAW_SLICE_FILES_FINAL,save_debug_images=rt.SAVE_DESCRIPTOR_DEBUG_IMAGES_FINAL,force=force,log_tag="final")
if not rt.RUN_DESCRIPTOR_FOR_SELECTED or (len(log) and not (log['status'].astype(str).str.upper()=='FAILED').any()): rt.save_stage_manifest(stage,"completed",[log_csv,rt.SELECTED_DESCRIPTOR_DIR] if rt.RUN_DESCRIPTOR_FOR_SELECTED else [],{"rows":len(log),"adaptive_gpu":True,"gpu_policy":rt.adaptive_gpu_snapshot().get("policy",{}),"cpu_worker_policy":[rt.DESCRIPTOR_CPU_WORKERS_MIN,rt.DESCRIPTOR_CPU_WORKERS_MAX]})
else: rt.save_stage_manifest(stage,"partial")
display(log.head() if len(log) else log)


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="12_final_master"; candidate_df=rt.load_df_checkpoint("candidate_parameters"); state=json.loads((rt.CHECKPOINT_DIR/"selection_state.json").read_text(encoding="utf-8")); ids=set(map(str,state['selected_ids'])); sub=candidate_df[candidate_df['candidate_id'].astype(str).isin(ids)].copy(); desc_log=rt.load_df_checkpoint("descriptor_log_final")
master,desc_cols,catalog=rt.collect_descriptor_master(sub,rt.SELECTED_DESCRIPTOR_DIR,rt.SELECTED_MASTER_XLSX,voxel_size_mm=rt.VOXEL_SIZE_MM,stage="final",run_log=desc_log); rt.save_df_checkpoint(master,"selected_final_master"); rt.atomic_json(rt.CHECKPOINT_DIR/"selected_descriptor_columns.json",desc_cols); rt.save_stage_manifest(stage,"completed",[rt.SELECTED_MASTER_XLSX,rt.SELECTED_MASTER_XLSX.with_suffix('.csv'),rt.CHECKPOINT_DIR/"selected_descriptor_columns.json"],{"rows":len(master),"descriptor_columns":len(desc_cols)}); display(master.head())


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
ptr=BASE_DIR/".ai_voxel_active_run.json"
if not ptr.exists(): raise FileNotFoundError("Run CELL 1 once: active run pointer missing.")
active=json.loads(ptr.read_text(encoding="utf-8")); RUN_DIR=Path(active["run_dir"])
os.environ["AI_VOXEL_ACTIVE_POINTER"]=str(ptr)
RUNTIME_DIR=RUN_DIR/"_runtime"
if str(RUNTIME_DIR) not in sys.path: sys.path.insert(0,str(RUNTIME_DIR))
import ai_voxel_runtime_dual3090_v4 as rt
rt=importlib.reload(rt)

stage="13_final_qa"; master=rt.load_df_checkpoint("selected_final_master"); report=rt.run_final_qa(); rt.save_stage_manifest(stage,"completed",[rt.STATE_DIR/"Final_QA.json"],{"final_rows":len(master),"final_columns":master.shape[1]}); print(json.dumps(report,indent=2,ensure_ascii=False)[:12000])
print("\nRUN_DIR:",rt.RUN_DIR); print("Final Excel:",rt.SELECTED_MASTER_XLSX); print("Final STL:",rt.SELECTED_STL_DIR); print("Final DLP:",rt.SELECTED_DLP_DIR); print("Final Descriptor:",rt.SELECTED_DESCRIPTOR_DIR)


In [ ]:
import os,sys,json,importlib
from pathlib import Path
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation");ptr=BASE_DIR/".ai_voxel_active_run.json"
active=json.loads(ptr.read_text(encoding='utf-8'));RUN_DIR=Path(active['run_dir']);os.environ['AI_VOXEL_ACTIVE_POINTER']=str(ptr)
RUNTIME_DIR=RUN_DIR/'_runtime';sys.path.insert(0,str(RUNTIME_DIR)) if str(RUNTIME_DIR) not in sys.path else None
import ai_voxel_runtime_dual3090_v4 as rt;rt=importlib.reload(rt)
stage='14_ml_dataset_contract'
pool=rt.load_df_checkpoint('pool_master');final=rt.load_df_checkpoint('selected_final_master',required=False)
contract,all_df,registry=rt.build_ml_dataset_contract(pool,final)
rt.save_stage_manifest(stage,'completed',[rt.ML_ROOT/'project_contract.json',rt.ML_ROOT/'all_structures.csv',rt.ML_ROOT/'data_registry.csv',rt.ML_ROOT/'experimental_specimen_manifest.csv'],{'rows':len(all_df),'registry_rows':len(registry)})
print(json.dumps(contract,indent=2,ensure_ascii=False));display(registry.head())
print('\nIMPORTANT: all non-LHS pool structures are retained in',rt.ML_ROOT/'all_structures.csv')


## 다음 단계
1. `01_Gen2Desc_Training_v1.ipynb`로 모든 생성구조를 학습합니다.
2. Descriptor-LHS로 선정한 final specimens를 DLP 제작/압축시험합니다.
3. 압축곡선은 `04_ml_dataset/compression_curve_long_template.csv` 형식이 가장 안정적이며, 기존 MEvoLattice v22 Excel 형식도 `02` Notebook adapter가 읽습니다.
